# Dashboard Qualité de Service — SNCB / Infrabel

**Auteur** : Tahar Guenfoud    
**Source** : [Open Data Infrabel](https://opendata.infrabel.be)

---

## Objectif
Construire un dashboard interactif pour analyser la ponctualité et la fiabilité du réseau ferroviaire belge.

## Plan du notebook
| Étape | Description |
|---|---|
| **1. Extract** | Téléchargement des 5 sources Open Data Infrabel |
| **2. Transform** | Nettoyage, typage, colonnes calculées |
| **3. EDA** | Analyse exploratoire — tendances, anomalies |
| **4. KPIs** | Calcul des indicateurs métier (Ponctualité, Reliability, Minutes perdues) |
| **5. Visualisation** | Graphiques pour le dashboard |

---
## 0. Imports & Configuration

In [2]:
import requests
import pandas as pd
import os
import time

# Dossier de sauvegarde
RAW_DIR   = "data/raw"
CLEAN_DIR = "data/clean"

print("✅ Imports OK")

✅ Imports OK


---
# ÉTAPE 1 — Extract
Téléchargement des 5 datasets depuis l'API Open Data Infrabel.

In [3]:
BASE = (
    "https://opendata.infrabel.be/api/explore/v2.1/catalog/datasets"
    "/{}/exports/csv?lang=fr&timezone=Europe%2FBrussels&use_labels=true&delimiter=%3B"
)

DATASETS = {
    "ponctualite_par_gare"    : "maandelijkse-stiptheid-per-stopplaats",
    "causes_retards"          : "oorzaken-vertraging-per-maand",
    "ponctualite_par_moment"  : "nationale-stiptheid-per-moment-en-per-maand",
    "trains_supprimes"        : "afgeschafte-treinen-per-maand-vanaf-2020",
    "kpi_contrat_performance" : "indicatoren-performantie-contract",
}

dfs = {}
for name, dataset_id in DATASETS.items():
    dfs[name] = pd.read_csv(BASE.format(dataset_id), sep=";")
    print(f"✅ {name:<35} {dfs[name].shape[0]:>6,} lignes × {dfs[name].shape[1]} cols")

✅ ponctualite_par_gare                27,343 lignes × 13 cols
✅ causes_retards                         425 lignes × 14 cols
✅ ponctualite_par_moment                 484 lignes × 9 cols
✅ trains_supprimes                        73 lignes × 7 cols
✅ kpi_contrat_performance                177 lignes × 13 cols


---
# ÉTAPE 2 — Transform
Nettoyage et préparation de chaque dataset.

### 2.1 — Ponctualité par Gare

In [12]:
# Explorer les colonnes brutes
df = dfs["ponctualite_par_gare"]
df.tail(10)

,Date,Point d'arrêt,Point d'arrêt.1,Point d'arrêt.2,ID point opérationnel,Classification,Classification.1,Classification.2,Ponctualité,Nombre total de trains,Nombre de trains ponctuels,Geo Point,Geo Shape
27333,2023-11,FRAMERIES,FRAMERIES,FRAMERIES,422,Stopplaats,Point d'arrêt,Stopping point,87.196468,906.0,790.0,"50.40552689016233, 3.906385857023495","{""coordinates"": [3.906385857023495, 50.4055268..."
27334,2023-11,FEXHE-LE-HAUT-CLOCHER,FEXHE-LE-HAUT-CLOCHER,FEXHE-LE-HAUT-CLOCHER,399,Stopplaats,Point d'arrêt,Stopping point,89.730290,964.0,865.0,"50.66429943269758, 5.397266518195518","{""coordinates"": [5.397266518195518, 50.6642994..."
27335,2023-11,FLEMALLE-GRANDE,FLEMALLE-GRANDE,FLEMALLE-GRANDE,401,Stopplaats,Point d'arrêt,Stopping point,91.383220,882.0,806.0,"50.605259633719655, 5.48108901393654","{""coordinates"": [5.48108901393654, 50.60525963..."
27336,2023-11,FLOREFFE,FLOREFFE,FLOREFFE,406,Stopplaats,Point d'arrêt,Stopping point,90.692124,1257.0,1140.0,"50.44348110123118, 4.762999764162857","{""coordinates"": [4.762999764162857, 50.4434811..."
27337,2023-11,FLORIVAL,FLORIVAL,FLORIVAL,410,Stopplaats,Point d'arrêt,Stopping point,94.306050,1405.0,1325.0,"50.76145683011463, 4.654204552903399","{""coordinates"": [4.654204552903399, 50.7614568..."
27338,2023-11,GONTRODE,GONTRODE,GONTRODE,474,Stopplaats,Point d'arrêt,Stopping point,79.947230,1137.0,909.0,"50.979840155563366, 3.801555019245788","{""coordinates"": [3.801555019245788, 50.9798401..."
27339,2023-11,FAUX,FAUX,FAUX,395,Stopplaats,Point d'arrêt,Stopping point,90.492170,894.0,809.0,"50.621786596713214, 4.549374449548871","{""coordinates"": [4.549374449548871, 50.6217865..."
27340,2023-11,BALEGEM-ZUID,BALEGEM-ZUID,BALEGEM-ZUID,106,Stopplaats,Point d'arrêt,Stopping point,79.525483,1138.0,905.0,"50.90080462540978, 3.805776391176711","{""coordinates"": [3.805776391176711, 50.9008046..."
27341,2023-11,AYE,AYE,AYE,100,Stopplaats,Point d'arrêt,Stopping point,95.246801,547.0,521.0,"50.22485292105391, 5.300638767374593","{""coordinates"": [5.300638767374593, 50.2248529..."
27342,2023-11,BAS-OHA,BAS-OHA,BAS-OHA,118,Stopplaats,Point d'arrêt,Stopping point,86.473430,828.0,716.0,"50.52248171944925, 5.191103060283717","{""coordinates"": [5.191103060283717, 50.5224817..."


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27343 entries, 0 to 27342
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Date                        27343 non-null  object 
 1   Point d'arrêt               27343 non-null  object 
 2   Point d'arrêt.1             27343 non-null  object 
 3   Point d'arrêt.2             27343 non-null  object 
 4   ID point opérationnel       27343 non-null  int64  
 5   Classification              27343 non-null  object 
 6   Classification.1            27343 non-null  object 
 7   Classification.2            27343 non-null  object 
 8   Ponctualité                 27343 non-null  float64
 9   Nombre total de trains      27343 non-null  float64
 10  Nombre de trains ponctuels  27343 non-null  float64
 11  Geo Point                   27307 non-null  object 
 12  Geo Shape                   27307 non-null  object 
dtypes: float64(3), int64(1), object

In [13]:
df.isnull().sum()

Date                           0
Point d'arrêt                  0
Point d'arrêt.1                0
Point d'arrêt.2                0
ID point opérationnel          0
Classification                 0
Classification.1               0
Classification.2               0
Ponctualité                    0
Nombre total de trains         0
Nombre de trains ponctuels     0
Geo Point                     36
Geo Shape                     36
dtype: int64

In [15]:
df["Date"].unique()

array(['2023-11', '2023-12', '2024-01', '2024-02', '2022-10', '2022-11',
       '2024-12', '2025-01', '2022-12', '2025-02', '2023-01', '2025-03',
       '2023-04', '2025-04', '2023-05', '2025-05', '2023-06', '2025-06',
       '2023-02', '2025-07', '2023-03', '2025-08', '2022-08', '2025-09',
       '2022-09', '2025-10', '2025-11', '2025-12', '2024-04', '2024-05',
       '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11',
       '2026-01', '2024-03', '2022-04', '2022-06', '2022-07', '2022-01',
       '2022-02', '2022-05', '2022-03', '2023-07', '2023-08', '2023-09',
       '2023-10'], dtype=object)

In [18]:
df_gare = dfs["ponctualite_par_gare"].copy()

# Renommer (13 colonnes dans l'ordre exact)
df_gare.columns = [
    "date",
    "nom_gare_fr",
    "nom_gare_nl",
    "nom_gare_de",
    "id_gare",
    "classification_fr",
    "classification_nl",
    "classification_de",
    "ponctualite_pct",
    "nb_trains",
    "nb_trains_ponctuels",
    "geo_point",
    "geo_shape"
]

# Parser la date
df_gare["date"] = pd.to_datetime(df_gare["date"], format="%Y-%m")

# Vérification
print(f"Shape : {df_gare.shape}")
print(f"Valeurs manquantes :\n{df_gare.isnull().sum()}")
df_gare.head(3)

Shape : (27343, 13)
Valeurs manquantes :
date                    0
nom_gare_fr             0
nom_gare_nl             0
nom_gare_de             0
id_gare                 0
classification_fr       0
classification_nl       0
classification_de       0
ponctualite_pct         0
nb_trains               0
nb_trains_ponctuels     0
geo_point              36
geo_shape              36
dtype: int64


,date,nom_gare_fr,nom_gare_nl,nom_gare_de,id_gare,classification_fr,classification_nl,classification_de,ponctualite_pct,nb_trains,nb_trains_ponctuels,geo_point,geo_shape
0,2023-11-01,BEIGNEE,BEIGNEE,BEIGNEE,133,Stopplaats,Point d'arrêt,Stopping point,91.200000,250.0,228.0,"50.33322928558822, 4.406633114707752","{""coordinates"": [4.406633114707752, 50.3332292..."
1,2023-11-01,BELSELE,BELSELE,BELSELE,138,Stopplaats,Point d'arrêt,Stopping point,83.060453,1588.0,1319.0,"51.15105293281537, 4.088971326009601","{""coordinates"": [4.088971326009601, 51.1510529..."
2,2023-11-01,BERLAAR,BERLAAR,BERLAAR,142,Stopplaats,Point d'arrêt,Stopping point,88.265746,1159.0,1023.0,"51.113708436411315, 4.638257976114972","{""coordinates"": [4.638257976114972, 51.1137084..."


In [19]:
# Voir quelques exemples côte à côte
df_gare[["nom_gare_fr", "nom_gare_nl", "nom_gare_de"]].drop_duplicates().head(20)

,nom_gare_fr,nom_gare_nl,nom_gare_de
0,BEIGNEE,BEIGNEE,BEIGNEE
1,BELSELE,BELSELE,BELSELE
2,BERLAAR,BERLAAR,BERLAAR
3,ANTWERPEN-LUCHTBAL,ANTWERPEN-LUCHTBAL,ANTWERPEN-LUCHTBAL
4,AARSELE,AARSELE,AARSELE
5,AISEAU,AISEAU,AISEAU
6,BAASRODE-ZUID,BAASRODE-ZUID,BAASRODE-ZUID
7,ANTWERPEN-ZUID,ANTWERPEN-ZUID,ANTWERPEN-ZUID
8,APPELTERRE,APPELTERRE,APPELTERRE
9,ARCHENNES,ARCHENNES,ARCHENNES


In [20]:
# Combien de fois les 3 colonnes sont identiques ?
identiques = (df_gare["nom_gare_fr"] == df_gare["nom_gare_nl"]).sum()
print(f"FR = NL : {identiques} fois sur {len(df_gare)}")

identiques2 = (df_gare["nom_gare_fr"] == df_gare["nom_gare_de"]).sum()
print(f"FR = DE : {identiques2} fois sur {len(df_gare)}")

FR = NL : 26046 fois sur 27343
FR = DE : 26046 fois sur 27343


In [21]:
# Supprimer les colonnes inutiles
df_gare = df_gare.drop(columns=["nom_gare_nl", "nom_gare_de",
                                 "classification_nl", "classification_de",
                                 "geo_shape"])

print(f"Colonnes restantes : {df_gare.columns.tolist()}")
print(f"Shape : {df_gare.shape}")

Colonnes restantes : ['date', 'nom_gare_fr', 'id_gare', 'classification_fr', 'ponctualite_pct', 'nb_trains', 'nb_trains_ponctuels', 'geo_point']
Shape : (27343, 8)


In [ ]:
display(df_gare.head())
df_gare.info()

### 2.2 — Causes des Retards

In [ ]:
# Explorer les colonnes brutes
df = dfs["causes_retards"]
print("Colonnes brutes :")
for col in df.columns:
    print(f"  • {col}")
df.head(3)

In [ ]:
df_causes = dfs["causes_retards"].copy()

# On adapte après avoir vu les colonnes ci-dessus
# TODO avec toi : renommer + parser la date

print(f"Shape : {df_causes.shape}")
print(f"Valeurs manquantes :\n{df_causes.isnull().sum()}")
df_causes.head(3)

### 2.3 — Ponctualité par Moment

In [ ]:
df = dfs["ponctualite_par_moment"]
print("Colonnes brutes :")
for col in df.columns:
    print(f"  • {col}")
df.head(3)

In [ ]:
df_moment = dfs["ponctualite_par_moment"].copy()

# TODO avec toi : renommer + parser la date + voir les moments disponibles

print(f"Shape : {df_moment.shape}")
df_moment.head(3)

### 2.4 — Trains Supprimés

In [ ]:
df = dfs["trains_supprimes"]
print("Colonnes brutes :")
for col in df.columns:
    print(f"  • {col}")
df.head(3)

In [ ]:
df_suppression = dfs["trains_supprimes"].copy()

# TODO avec toi : renommer + parser la date

print(f"Shape : {df_suppression.shape}")
df_suppression.head(3)

### 2.5 — KPIs Contrat de Performance

In [ ]:
df = dfs["kpi_contrat_performance"]
print("Colonnes brutes :")
for col in df.columns:
    print(f"  • {col}")
df.head(3)

In [ ]:
df_kpi = dfs["kpi_contrat_performance"].copy()

# TODO avec toi : filtrer les KPIs pertinents pour la ponctualité

print(f"Shape : {df_kpi.shape}")
print(f"\nCatégories disponibles :")
# df_kpi["Catégorie"].value_counts()
df_kpi.head(5)

---
# ÉTAPE 3 — EDA
> 🔜 À compléter ensemble après le Transform

---
# ÉTAPE 4 — KPIs Métier
> 🔜 À compléter ensemble après l'EDA

---
# ÉTAPE 5 — Visualisation
> 🔜 À compléter ensemble après les KPIs